In [64]:
import pandas as pd
import numpy as np
from pyspark.sql.functions import col, count, sum as _sum, abs as _abs, isnan

StatementMeta(, 356b8bdc-5a34-4c25-a746-2cd7bf6120d1, 66, Finished, Available, Finished)

In [65]:
payments = pd.read_csv("/lakehouse/default/Files/olist_order_payments_dataset.csv")
print(payments.head())

StatementMeta(, 356b8bdc-5a34-4c25-a746-2cd7bf6120d1, 67, Finished, Available, Finished)

                           order_id  payment_sequential payment_type  \
0  b81ef226f3fe1789b1e8b2acac839d17                   1  credit_card   
1  a9810da82917af2d9aefd1278f1dcfa0                   1  credit_card   
2  25e8ea4e93396b6fa0d3dd708e76c1bd                   1  credit_card   
3  ba78997921bbcdc1373bb41e913ab953                   1  credit_card   
4  42fdf880ba16b47b59251dd489d4441a                   1  credit_card   

   payment_installments  payment_value  
0                     8          99.33  
1                     1          24.39  
2                     1          65.71  
3                     8         107.78  
4                     2         128.45  


In [66]:
print(payments.info())

StatementMeta(, 356b8bdc-5a34-4c25-a746-2cd7bf6120d1, 68, Finished, Available, Finished)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB
None


In [67]:
profile = pd.DataFrame({
    'Column': payments.columns.values,
    'negative(%)': [
        len(payments[col][payments[col] < 0]) / len(payments) * 100 if col in payments.select_dtypes(include=[np.number]).columns else 0
        for col in payments.columns
    ],  
    'zero(%)': [
        len(payments[col][payments[col] == 0]) / len(payments) * 100 if col in payments.select_dtypes(include=[np.number]).columns else 0
        for col in payments.columns
    ],  
    'duplicates': payments.duplicated().sum(), 
    'unique': payments.nunique().values, 
})

profile

StatementMeta(, 356b8bdc-5a34-4c25-a746-2cd7bf6120d1, 69, Finished, Available, Finished)

,Column,negative(%),zero(%),duplicates,unique
0,order_id,0.0,0.000000,0,99440
1,payment_sequential,0.0,0.000000,0,29
2,payment_type,0.0,0.000000,0,5
3,payment_installments,0.0,0.001925,0,24
4,payment_value,0.0,0.008663,0,29077


In [68]:
print(payments[payments["payment_installments"] == 0])

StatementMeta(, 356b8bdc-5a34-4c25-a746-2cd7bf6120d1, 70, Finished, Available, Finished)

                               order_id  payment_sequential payment_type  \
46982  744bade1fcf9ff3f31d860ace076d422                   2  credit_card   
79014  1a57108394169c0b47d8f876acc9ba2d                   2  credit_card   

       payment_installments  payment_value  
46982                     0          58.69  
79014                     0         129.94  


In [69]:
print(payments[payments["payment_type"] == 'not_defined'])

StatementMeta(, 356b8bdc-5a34-4c25-a746-2cd7bf6120d1, 71, Finished, Available, Finished)

                               order_id  payment_sequential payment_type  \
51280  4637ca194b6387e2d538dc89b124b0ee                   1  not_defined   
57411  00b1cb0320190ca0daa2c88b35206009                   1  not_defined   
94427  c8c528189310eaa44a745b8d9d26908b                   1  not_defined   

       payment_installments  payment_value  
51280                     1            0.0  
57411                     1            0.0  
94427                     1            0.0  


Check for order status of payment_type that is 'not_defined.

In [70]:
silver_orders= spark.read.table("SilverLakehouse.dbo.olist_orders_cleaned")

StatementMeta(, 356b8bdc-5a34-4c25-a746-2cd7bf6120d1, 72, Finished, Available, Finished)

In [71]:
# Find rows with undefined payment type
undefined_payments = payments[payments["payment_type"] == "not_defined"]

# Convert silver_orders to a dataframe
orders = silver_orders.toPandas()

# Left join with orders table
undefined_with_orders = pd.merge(
    undefined_payments,
    orders,
    on="order_id",       # common column
    how="left"           # left join keeps all from undefined_payments
)

# Display results
print(undefined_with_orders[["order_id", "payment_type", "order_status"]])


StatementMeta(, 356b8bdc-5a34-4c25-a746-2cd7bf6120d1, 73, Finished, Available, Finished)

                           order_id payment_type order_status
0  4637ca194b6387e2d538dc89b124b0ee  not_defined     canceled
1  00b1cb0320190ca0daa2c88b35206009  not_defined     canceled
2  c8c528189310eaa44a745b8d9d26908b  not_defined     canceled


Since, all of the orders of the not_defined payment_type are canceled, we can drop these rows. 

In [72]:
payments.drop(payments[payments["payment_type"] == "not_defined"].index, inplace=True)

print(payments.info())

StatementMeta(, 356b8bdc-5a34-4c25-a746-2cd7bf6120d1, 74, Finished, Available, Finished)

<class 'pandas.core.frame.DataFrame'>
Index: 103883 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103883 non-null  object 
 1   payment_sequential    103883 non-null  int64  
 2   payment_type          103883 non-null  object 
 3   payment_installments  103883 non-null  int64  
 4   payment_value         103883 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.8+ MB
None


In [73]:
# Check post-cleaned dataframe

assert payments['payment_type'].isnull().sum() == 0, "Null values found in payment_type"
assert payments['order_id'].isnull().sum() == 0, "Null values found in order id"
assert payments['payment_type'].isnull().sum() == 0, "Null values found in payment_type"
assert payments['payment_value'].isnull().sum() == 0, "Null values found in payment_value"
assert payments['payment_installments'].isnull().sum() == 0, "Null values found in payment_instalments"

assert 'not_defined' not in payments['payment_type'].values, f"Value {value_to_check} found in column 'column_name'"


StatementMeta(, 356b8bdc-5a34-4c25-a746-2cd7bf6120d1, 75, Finished, Available, Finished)

In [74]:
# Find rows with undefined payment type
zero_payment_installments = payments[payments["payment_installments"] == 0]


# Left join with orders table
zero_with_orders = pd.merge(
    zero_payment_installments,
    orders,
    on="order_id",
    how="left"           
)

# Display results
print(zero_with_orders[["order_id", "payment_installments", "payment_type", "order_status"]])

StatementMeta(, 356b8bdc-5a34-4c25-a746-2cd7bf6120d1, 76, Finished, Available, Finished)

                           order_id  payment_installments payment_type  \
0  744bade1fcf9ff3f31d860ace076d422                     0  credit_card   
1  1a57108394169c0b47d8f876acc9ba2d                     0  credit_card   

  order_status  
0    delivered  
1    delivered  


Since, the order status was delivered, and all other credit card orders have at least a payment installment value of 1. Let's check the payment value with the order_items price and freight value too. 

In [75]:
silver_items= spark.read.table("SilverLakehouse.dbo.olist_items_cleaned")

StatementMeta(, 356b8bdc-5a34-4c25-a746-2cd7bf6120d1, 77, Finished, Available, Finished)

In [76]:
# Convert silver_items to a dataframe
items = silver_items.toPandas()

StatementMeta(, 356b8bdc-5a34-4c25-a746-2cd7bf6120d1, 78, Finished, Available, Finished)

In [77]:
# Left join with items table
zero_with_price = pd.merge(
    zero_with_orders,
    items,
    on="order_id",
    how="left"           
)

zero_with_price["order_total"] = zero_with_price["price"] + zero_with_price ["freight_value"]

# Display results
print(zero_with_price[["order_id", "order_item_id", "payment_installments", "payment_type", "order_status", "payment_value", "price", "freight_value", "order_total"]])

StatementMeta(, 356b8bdc-5a34-4c25-a746-2cd7bf6120d1, 79, Finished, Available, Finished)

                           order_id  order_item_id  payment_installments  \
0  744bade1fcf9ff3f31d860ace076d422              1                     0   
1  1a57108394169c0b47d8f876acc9ba2d              1                     0   
2  1a57108394169c0b47d8f876acc9ba2d              2                     0   

  payment_type order_status  payment_value  price  freight_value  order_total  
0  credit_card    delivered          58.69  45.90          12.79        58.69  
1  credit_card    delivered         129.94  41.69          23.28        64.97  
2  credit_card    delivered         129.94  41.69          23.28        64.97  


Since, for both of these orders the payment values match the order items table and their status is shown as delivered, for consistency, their payment installments will be changed to 1.

In [78]:
# For every row where payment_installment is less than 1 (ie. 0), replace the value with 1
payments.loc[payments['payment_installments'] < 1, 'payment_installments'] = 1

StatementMeta(, 356b8bdc-5a34-4c25-a746-2cd7bf6120d1, 80, Finished, Available, Finished)

In [79]:
# Check dataframe does not have payment_installment of 0
assert (payments["payment_installments"] != 0).all(), "There are rows with payment_installments = 0!"

StatementMeta(, 356b8bdc-5a34-4c25-a746-2cd7bf6120d1, 81, Finished, Available, Finished)

In [80]:
# Write the table to the silver lakehouse as a delta table
# Convert pandas to Spark
spark_payments = spark.createDataFrame(payments)

# Save as a Delta table in Silver Lakehouse
silver_path = "SilverLakehouse.dbo.olist_payments_cleaned"
spark_payments.write.mode("overwrite").format("delta").saveAsTable(silver_path)

StatementMeta(, 356b8bdc-5a34-4c25-a746-2cd7bf6120d1, 82, Finished, Available, Finished)